In [1]:
#Task 3
# Tek bir self-attention head yaz: query, key, value ve pozisyon embedding'i.
# Bir örnekte ağırlık matrisini (wei) yazdır, bir satırını oku: o token hangi
# token'lara ne kadar bakıyor. Skorları head_size'ın kareköküne neden
# böldüğümüzü sayıyla göster: bölmeden ve bölerek softmax çıktısını yan yana
# koy. (58:26 - 1:19:11)

In [2]:
import torch 
from torch.nn import functional as F
import torch.nn as nn
torch.manual_seed(1337)
B,T,C = 4,8,32
token_emb = torch.randn(B,T,C)
pos_emb = torch.randn(T,C)
x = token_emb + pos_emb


In [3]:
headsize= 16
torch.manual_seed(1337)

#defining them as linear layer without bias so typically just doing matrix mul
#class torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None) default linear class y = xA^T
key = nn.Linear(C, headsize, bias=False)   #outputs shape [4, 8, 16]
query = nn.Linear(C, headsize, bias=False) #outputs shape [4, 8, 16]
value = nn.Linear(C, headsize, bias=False) #outputs shape [4, 8, 16]

k = key(x)   
q = query(x)
wei = q @ k.transpose(-2,-1)# (4,8,16) @ (4,16,8) = (4,8,8) pytorch understands its batch matrix mull so multiplying them parallel and looking for last 2 dimension.
wei = wei*headsize**-0.5

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei,dim=-1)

v = value(x)
out = wei @ v


In [4]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4143, 0.5857, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1236, 0.6373, 0.2391, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1856, 0.2091, 0.4027, 0.2026, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0602, 0.1894, 0.4719, 0.2195, 0.0589, 0.0000, 0.0000, 0.0000],
        [0.1427, 0.1791, 0.2383, 0.2483, 0.1056, 0.0860, 0.0000, 0.0000],
        [0.0492, 0.1025, 0.0442, 0.1770, 0.5202, 0.0603, 0.0466, 0.0000],
        [0.1950, 0.0933, 0.0525, 0.0342, 0.1837, 0.2506, 0.1225, 0.0682]],
       grad_fn=<SelectBackward0>)

In [5]:
headsize= 16
torch.manual_seed(1337)
#defining them as linear layer without bias so typically just doing matrix mul
#class torch.nn.Linear(in_features, out_features, bias=True, device=None, dtype=None) default linear class y = xA^T
key = nn.Linear(C, headsize, bias=False)   #outputs shape [4, 8, 16]
query = nn.Linear(C, headsize, bias=False) #outputs shape [4, 8, 16]
value = nn.Linear(C, headsize, bias=False) #outputs shape [4, 8, 16]

k = key(x)   
q = query(x)
wei = q @ k.transpose(-2,-1) # (4,8,16) @ (4,16,8) = (4,8,8) pytorch understands its batch matrix mull so multiplying them parallel and looking for last 2 dimension.

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei,dim=-1)

v = value(x)
out = wei @ v
wei[0]

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [2.0020e-01, 7.9980e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.3880e-03, 9.7921e-01, 1.9406e-02, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [3.8156e-02, 6.1559e-02, 8.4610e-01, 5.4189e-02, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [2.4643e-04, 2.4191e-02, 9.3171e-01, 4.3624e-02, 2.2601e-04, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [4.7947e-02, 1.1908e-01, 3.7273e-01, 4.3954e-01, 1.4380e-02, 6.3238e-03,
         0.0000e+00, 0.0000e+00],
        [7.8754e-05, 1.4845e-03, 5.1184e-05, 1.3208e-02, 9.8494e-01, 1.7830e-04,
         6.3292e-05, 0.0000e+00],
        [2.1079e-01, 1.1057e-02, 1.1031e-03, 1.9851e-04, 1.6590e-01, 5.7493e-01,
         3.2853e-02, 3.1598e-03]], grad_fn=<SelectBackward0>)

In [6]:
#why are scaling them with 1/root(headsize)
torch.manual_seed(1337)
a = torch.randn(T,headsize)
b = torch.randn(T,headsize)
c = a @ b.T


In [7]:
a.var(), b.var()

(tensor(1.0797), tensor(0.9995))

In [8]:
c.var()

tensor(15.7589)

In [9]:
d = a @ b.T * headsize**-0.5

In [10]:
d.var()

tensor(0.9849)

In [11]:
#softmax converges to produce one hot vector when its inputs are too spreaded thats why we want to keep their var around 1